# CreditWise Loan Approval System
### SecureTrust Bank — Intelligent Loan Approval Prediction

**Goal:** Predict whether a loan application should be `Approved (1)` or `Rejected (0)` using historical applicant data, replacing an inconsistent manual verification process.

**Your role in this project:** the pipeline below (data loading → cleaning → EDA → preprocessing → baseline models) is built for you. Sections marked **`### YOUR TURN`** are where you take over — hyperparameter tuning, model comparison, and error analysis. That's the part that will actually teach you the most, so don't skip it.

**Dataset note:** This is a *synthetic* dataset generated to match the column schema from the problem statement, with realistic relationships baked in (credit score, income, DTI ratio, collateral, and existing loans all influence approval, plus some noise to mimic real-world inconsistency). It also has a few missing values injected on purpose so you get practice handling them.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

df = pd.read_csv('creditwise_loan_data.csv')
print(df.shape)
df.head()


## 1. Exploratory Data Analysis (EDA)

In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
# Missing values
df.isnull().sum()[df.isnull().sum() > 0]


In [ ]:
# Target distribution
df['Loan_Approved'].value_counts(normalize=True).plot(kind='bar', title='Loan_Approved distribution')
plt.xlabel('Loan_Approved (1=Approved, 0=Rejected)')
plt.ylabel('Proportion')
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.boxplot(data=df, x='Loan_Approved', y='Credit_Score', ax=axes[0,0])
sns.boxplot(data=df, x='Loan_Approved', y='DTI_Ratio', ax=axes[0,1])
sns.boxplot(data=df, x='Loan_Approved', y='Applicant_Income', ax=axes[1,0])
sns.boxplot(data=df, x='Loan_Approved', y='Existing_Loans', ax=axes[1,1])
plt.tight_layout()
plt.show()


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).drop(columns=['Loan_Approved']).columns
corr = df[list(numeric_cols) + ['Loan_Approved']].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation with Loan_Approved')
plt.show()


**What to look for here:** which numeric features correlate most with `Loan_Approved`? In this dataset, `Credit_Score` should stand out, followed by income and `DTI_Ratio`. Keep this in mind later when you inspect feature importances — do they agree?

## 2. Data Cleaning & Feature Engineering

In [ ]:
df_clean = df.copy()

# Drop the ID column - it's not predictive
df_clean = df_clean.drop(columns=['Applicant_ID'])

# Impute missing numeric values with median (robust to outliers)
for col in ['Dependents', 'Credit_Score', 'Savings', 'Collateral_Value']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Feature engineering: total household income is often more predictive than applicant income alone
df_clean['Total_Income'] = df_clean['Applicant_Income'] + df_clean['Coapplicant_Income']
df_clean['Loan_to_Income_Ratio'] = df_clean['Loan_Amount'] / (df_clean['Total_Income'] + 1)

df_clean.isnull().sum().sum()  # should be 0


## 3. Encoding & Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

categorical_cols = ['Employment_Status', 'Marital_Status', 'Loan_Purpose',
                     'Property_Area', 'Education_Level', 'Gender', 'Employer_Category']

df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

X = df_encoded.drop(columns=['Loan_Approved'])
y = df_encoded['Loan_Approved']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numeric features (helps Logistic Regression; tree models don't need it but it doesn't hurt)
scaler = StandardScaler()
numeric_features = ['Applicant_Income','Coapplicant_Income','Age','Dependents','Credit_Score',
                     'Existing_Loans','DTI_Ratio','Savings','Collateral_Value','Loan_Amount',
                     'Loan_Term','Total_Income','Loan_to_Income_Ratio']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test[numeric_features])

print(X_train.shape, X_test.shape)


## 4. Baseline Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, classification_report)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
}

results = []
for name, model in models.items():
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
        probs = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1': f1_score(y_test, preds),
        'ROC_AUC': roc_auc_score(y_test, probs)
    })

results_df = pd.DataFrame(results).set_index('Model').round(3)
results_df


**Reading these metrics for a loan approval system:**
- **Precision** (of predicted approvals, how many were correctly approved) matters if false approvals are costly — that's the "high-risk customer gets approved" problem from the brief.
- **Recall** (of actual good applicants, how many did we correctly approve) matters if false rejections are costly — that's the "good customer gets rejected" problem.
- There's a real business trade-off here between the two. Which one should SecureTrust Bank prioritize? There's no single right answer — think about it in terms of cost of a bad loan vs. cost of a lost customer.


---
## 5. `### YOUR TURN` — Tuning & Analysis

This is the section to make your own. Some concrete things to try, roughly in order of value for a first project:

1. **Hyperparameter tuning.** Use `GridSearchCV` or `RandomizedSearchCV` on the Random Forest (e.g. `n_estimators`, `max_depth`, `min_samples_split`) and/or try `XGBoost` / `GradientBoostingClassifier`. Compare against the baselines above.
2. **Feature importance.** Plot `model.feature_importances_` for your best tree-based model. Does it match what you saw in the correlation heatmap in Section 1?
3. **Confusion matrix deep-dive.** Plot it with `ConfusionMatrixDisplay`. Look specifically at false positives (bad loans approved) vs false negatives (good customers rejected) — which does your best model favor, and does that match what the bank would want?
4. **Threshold tuning.** Instead of the default 0.5 cutoff on `predict_proba`, try shifting the decision threshold and see how precision/recall trade off. Plot a precision-recall curve.
5. **Cross-validation.** Use `cross_val_score` (e.g. 5-fold) instead of a single train/test split to get a more reliable estimate of model performance.
6. **(Stretch) Class imbalance handling.** This dataset is fairly balanced, but in practice loan datasets often aren't — look into `class_weight='balanced'` or `SMOTE` and see how they affect precision/recall.

Add your code in the cells below.


In [ ]:
# 1. Hyperparameter tuning
from sklearn.model_selection import GridSearchCV

# Example starting point - expand the grid as you experiment
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

# grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='f1')
# grid_search.fit(X_train, y_train)
# print(grid_search.best_params_)


In [ ]:
# 2. Feature importance
# best_rf = grid_search.best_estimator_
# importances = pd.Series(best_rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
# importances.head(15).plot(kind='barh')
# plt.gca().invert_yaxis()
# plt.title('Top 15 Feature Importances')
# plt.show()


In [ ]:
# 3. Confusion matrix deep-dive
# from sklearn.metrics import ConfusionMatrixDisplay
# ConfusionMatrixDisplay.from_estimator(best_rf, X_test, y_test)
# plt.show()


---
## 6. Next Steps / Ideas to Extend This Project
- Deploy the final model behind a simple Streamlit or Flask app where a loan officer inputs applicant details and gets a prediction + confidence score.
- Add SHAP values for individual-prediction explainability (important for regulated domains like lending — you generally can't just say "the model said no").
- Test for fairness/bias across `Gender`, `Property_Area`, or `Employer_Category` — does the model approve at different rates across these groups even when controlling for creditworthiness? This directly addresses the "biased" complaint in the original problem statement.
